# Light GBM

LightGBM (Light Gradient Boosting Machine) is a fast, high-performance gradient boosting framework developed by Microsoft that builds decision trees sequentially to predict continuous numerical values in supervised regression

# More infromation below the link

https://docs.google.com/document/d/1sfqFsNs7xyPdszpo5GhY1tJ_eX9d3PK408o5M4-PyDc/edit?usp=sharing

# Notes and interview Question and answer

https://docs.google.com/document/d/1KPo4dbcnzlk-_TQzfOqxr5K_adaxNeRS5lN2uiDObwU/edit?usp=sharing

# Coding

In [1]:
import lightgbm as lgb  # LightGBM → gradient boosting framework

print(lgb.__version__)  # Shows installed LightGBM version

4.7.0


In [2]:

import numpy as np  # NumPy → numerical calculations
import pandas as pd  # Pandas → data manipulation

from sklearn.datasets import fetch_california_housing  # California Housing → built-in regression dataset

from sklearn.model_selection import train_test_split  # Splits data into training and testing sets

from lightgbm import LGBMRegressor  # LGBMRegressor → LightGBM regression algorithm

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Regression evaluation metrics

In [3]:
# load california Housing 

housing = fetch_california_housing(as_frame=True)  # Loads California Housing as a Pandas DataFrame

X = housing.data  # X → input features used to predict house values

y = housing.target  # y → target values we want the model to predict

In [4]:
print(X.shape)  # Shows number of samples and features

print(y.shape)  # Shows number of target values

X.head()  # Displays the first five rows of features

(20640, 8)
(20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [5]:
# Train  / Test Split 

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                    # X = features, y = target
    test_size=0.2,            # 20% → testing data, 80% → training data
    random_state=42           # Keeps the same split every time
)

In [6]:
# Create LightGBM

lgb_model = LGBMRegressor(
    n_estimators=300,       # Number of boosting trees
    learning_rate=0.05,     # Contribution of each tree
    num_leaves=31,          # Maximum number of leaves in each tree; controls model complexity
    max_depth=-1,            # -1 → no explicit maximum depth
    subsample=0.8,           # Uses 80% of training samples for each boosting iteration
    colsample_bytree=0.8,    # Uses 80% of features for each tree
    reg_alpha=0.0,           # L1 regularization strength
    reg_lambda=0.0,          # L2 regularization strength
    random_state=42,         # Makes results reproducible
    n_jobs=-1                # Uses all available CPU cores
)

In [7]:
# Train LightGBM
lgb_model.fit(
    X_train,  # Training features
    y_train   # Training target values
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001879 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 16512, number of used features: 8
[LightGBM] [Info] Start training from score 2.071947


,learning_rate,0.05
,n_estimators,300
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None


In [8]:
# Make Predictions

y_pred = lgb_model.predict(
    X_test  # Unseen test features used to generate predictions
)

In [9]:
# Evaluate 

rmse = np.sqrt(mean_squared_error(
    y_test,  # Actual target values
    y_pred   # LightGBM predictions
))

mae = mean_absolute_error(
    y_test,  # Actual target values
    y_pred   # LightGBM predictions
)

r2 = r2_score(
    y_test,  # Actual target values
    y_pred   # LightGBM predictions
)

In [10]:
print("RMSE:", rmse)  # RMSE → lower is better

print("MAE:", mae)  # MAE → lower is better

print("R²:", r2)  # R² → higher is better

RMSE: 0.4465263286941419
MAE: 0.295255996053389
R²: 0.8478446305483


In [11]:
# Hyperparameter tuning 

# Now we will tune the most important LightGBM parameters. 


from sklearn.model_selection import GridSearchCV  # GridSearchCV → tests different hyperparameter combinations using cross-validation

In [14]:
param_grid = {
    "n_estimators": [200, 300],  # Tests the number of boosting trees
    "learning_rate": [0.05, 0.1],  # Tests the contribution of each tree
    "num_leaves": [15, 31, 63],  # Tests the complexity of leaf-wise trees
    "min_child_samples": [10, 20],  # Controls minimum samples in each leaf
    "subsample": [0.8, 1.0],  # Tests the fraction of training samples used
    "colsample_bytree": [0.8, 1.0]  # Tests the fraction of features used
}

# parameters value n_estimators to colsample_bytree

# 2 × 2 × 3 × 2 × 2 × 2 = 96 combinations
# with 5 fold cv  , 96*5 = 480 model fits
# much more resonable 



In [15]:
# GridSearch 

grid_search = GridSearchCV(
    estimator=lgb_model,  # The LightGBM model whose hyperparameters we want to optimize
    param_grid=param_grid,  # Hyperparameter combinations GridSearchCV will test
    cv=5,  # 5-fold cross-validation → each training portion becomes validation once
    scoring="neg_root_mean_squared_error",  # Uses RMSE; sklearn makes it negative because GridSearchCV maximizes scores
    n_jobs=-1  # Uses all available CPU cores to speed up the search
)

In [16]:
# Run Hyperparameter search 


grid_search.fit(
    X_train,  # Training features used during cross-validation
    y_train   # Training target values used during cross-validation
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003404 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 13210, number of used features: 8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005708 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005931 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1837
[LightGBM] [Info] Number of data points in the train set: 13209, number of used features: 8
[LightGBM] [Info] Number of data points in the train set: 13210, number of used features: 8
[LightGBM] [Info] Start training from score 2.070050
[LightGBM] [Info] Start training from score 2.067432
[LightGBM] [Info] Start traini

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMRegressor...subsample=0.8)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.05, 0.1], 'min_child_samples': [10, 20], 'n_estimators': [200, 300], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for ca

In [17]:
# Find the best parameters 
print(
    "Best Parameters:",
    grid_search.best_params_  # Shows the parameter combination with the best cross-validation performance
)

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'min_child_samples': 20, 'n_estimators': 300, 'num_leaves': 63, 'subsample': 0.8}


In [18]:
# get the best model 


best_lgb = grid_search.best_estimator_  # Retrieves the best LightGBM model found by GridSearchCV

In [19]:
# Final prediction 
y_pred_best = best_lgb.predict(
    X_test  # Completely unseen test features
)

In [20]:
# Final Evaluation 


final_rmse = np.sqrt(mean_squared_error(
    y_test,  # Actual test target values
    y_pred_best  # Predictions from the tuned LightGBM model
))

final_mae = mean_absolute_error(
    y_test,  # Actual test target values
    y_pred_best  # Predictions from the tuned LightGBM model
)

final_r2 = r2_score(
    y_test,  # Actual test target values
    y_pred_best  # Predictions from the tuned LightGBM model
)

In [21]:
print("Final RMSE:", final_rmse)  # Lower → better prediction error

print("Final MAE:", final_mae)  # Lower → better average error

print("Final R²:", final_r2)  # Higher → better explained variance

Final RMSE: 0.4350225109303739
Final MAE: 0.28436996755501054
Final R²: 0.8555835711151718
